# Panatang Makabayan Detector
## Tandem Image + Audio Preprocessing Pipeline

This notebook implements a synchronized preprocessing pipeline for:
- **11 JPEG/PNG frames** from 10-second video
- **1 WAV audio file** synchronized to frames
- **Feature extraction**: Image + Audio → fused 17-D features per timestamp
- **Output**: Edge images, audio segments, feature arrays, timeline CSV

Uses OpenCV (image), librosa (audio), and NumPy (feature fusion).

## 1. Environment Setup

In [7]:
!pip install -q librosa matplotlib

import cv2
import numpy as np
import librosa
import librosa.display
import os
import csv
import warnings
warnings.filterwarnings('ignore')

print(f'OpenCV  : {cv2.__version__}')
print(f'NumPy   : {np.__version__}')
print(f'librosa : {librosa.__version__}')
print('✓ All imports OK')

OpenCV  : 4.13.0
NumPy   : 2.4.4
librosa : 0.11.0
✓ All imports OK



[notice] A new release of pip is available: 24.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import cv2
import numpy as np
import librosa
import librosa.display
import os
import csv
import warnings
warnings.filterwarnings('ignore')

print(f'OpenCV  : {cv2.__version__}')
print(f'NumPy   : {np.__version__}')
print(f'librosa : {librosa.__version__}')
print('✓ All imports OK')

OpenCV  : 4.13.0
NumPy   : 2.4.4
librosa : 0.11.0
✓ All imports OK


## 2. Configuration

In [9]:
# ── Paths ─────────────────────────────────────────────────────────────────────
IMAGE_FOLDER  = 'Jupyter/frames'
AUDIO_FILE    = 'PANATANG MAKABAYAN.wav'
OUTPUT_FOLDER = 'sample_preprocessed'

# ── Parameters ─────────────────────────────────────────────────────────────────
NUM_FRAMES    = 11
VIDEO_SECONDS = 10
BLUR_KERNEL   = (5, 5)
CANNY_LOW     = 50
CANNY_HIGH    = 150
TARGET_W, TARGET_H = 640, 480

# ── Audio Parameters ───────────────────────────────────────────────────────────
TARGET_SR     = 22050
N_MFCC        = 13

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(f'{OUTPUT_FOLDER}/frames_edges', exist_ok=True)
os.makedirs(f'{OUTPUT_FOLDER}/audio_segments', exist_ok=True)
os.makedirs(f'{OUTPUT_FOLDER}/features', exist_ok=True)
print('✓ Output directories created')

✓ Output directories created


## 3. Data Loading

### 3.1 Load 11 Images

In [10]:
def load_frames(folder: str, n: int = NUM_FRAMES):
    """Load N frames from folder (1-based indexing: frame_01, frame_02, ...)"""
    frames = []
    for i in range(n):
        frame_num = i + 1  # Convert 0-based to 1-based (0->1, 1->2, ..., 10->11)
        candidates = [
            f'{folder}/frame_{frame_num:02d}.jpg',
            f'{folder}/frame_{frame_num:02d}.png',
            f'{folder}/frame_{frame_num}.jpg',
            f'{folder}/frame_{frame_num}.png',
        ]
        loaded = None
        for path in candidates:
            if os.path.exists(path):
                loaded = cv2.imread(path)
                if loaded is not None:
                    print(f'  [{i:02d}] Loaded {os.path.basename(path)}')
                    break
        if loaded is None:
            print(f'  [{i:02d}] ⚠ Not found — using placeholder')
            loaded = np.random.randint(80, 200, (480, 640, 3), dtype=np.uint8)
        frames.append(loaded)
    return frames

frames_raw = load_frames(IMAGE_FOLDER)
print(f'\n✓ Loaded {len(frames_raw)} frames')

  [00] ⚠ Not found — using placeholder
  [01] ⚠ Not found — using placeholder
  [02] ⚠ Not found — using placeholder
  [03] ⚠ Not found — using placeholder
  [04] ⚠ Not found — using placeholder
  [05] ⚠ Not found — using placeholder
  [06] ⚠ Not found — using placeholder
  [07] ⚠ Not found — using placeholder
  [08] ⚠ Not found — using placeholder
  [09] ⚠ Not found — using placeholder
  [10] ⚠ Not found — using placeholder

✓ Loaded 11 frames


### 3.2 Load WAV Audio (OpenCV-compatible via `wave` stdlib)

In [11]:
if os.path.exists(AUDIO_FILE):
    audio_raw, sr_orig = librosa.load(AUDIO_FILE, sr=None)
    print(f'✓ Audio loaded  sr={sr_orig} Hz  duration={len(audio_raw)/sr_orig:.2f}s')
else:
    print('⚠ Audio file not found — using synthetic signal')
    sr_orig = 44100
    t = np.linspace(0, VIDEO_SECONDS, sr_orig * VIDEO_SECONDS)
    audio_raw = (0.5 * np.sin(2 * np.pi * 220 * t)).astype(np.float32)
    print(f'✓ Synthetic audio created  sr={sr_orig}')

✓ Audio loaded  sr=44100 Hz  duration=10.03s


---
## 4. Image Preprocessing Pipeline (OpenCV)

### Engraved Plan

| Step | Operation | OpenCV call | Purpose |
|------|-----------|-------------|---------|
| 1 | **Noise Reduction** | `cv2.GaussianBlur(frame, (5,5), 0)` | Suppress high-frequency sensor noise and lighting flicker |
| 2 | **Color-Space Conversion** | `cv2.cvtColor(blurred, cv2.COLOR_BGR2GRAY)` | Reduce 3-channel colour redundancy; grayscale is sufficient for motion/edge analysis |
| 3 | **Normalization** | Min-Max: `(x − xmin) / (xmax − xmin)` | Equalise per-frame exposure; output ∈ [0, 1] |
| 4a | **Edge Features** | `cv2.Canny(norm_u8, 50, 150)` | Capture hand/lip boundary sharpness (proxy for articulation confidence) |
| 4b | **Texture Features** | Bitwise LBP approximation | Encode micro-texture (skin, fabric) without external libs |
| 4c | **Shape Features** | `cv2.findContours` + `cv2.moments` | Extract centroid, area, aspect ratio of dominant blobs |

In [12]:
def preprocess_image(frame: np.ndarray) -> dict:
    """
    Image preprocessing:
    1. Gaussian Blur (noise reduction)
    2. Grayscale conversion
    3. Min-Max normalization
    4. Canny edge detection
    5. Feature extraction from edges
    """
    # Step 1: Blur
    blurred = cv2.GaussianBlur(frame, BLUR_KERNEL, sigmaX=0)
    
    # Step 2: Grayscale
    gray = cv2.cvtColor(blurred, cv2.COLOR_BGR2GRAY)
    
    # Step 3: Normalize [0,1]
    x_min, x_max = gray.min(), gray.max()
    normalized = (gray.astype(np.float32) - x_min) / (max(x_max - x_min, 1e-8))
    norm_u8 = (normalized * 255).astype(np.uint8)
    
    # Step 4: Edge detection
    edges = cv2.Canny(norm_u8, CANNY_LOW, CANNY_HIGH)
    
    # Step 5: Extract shape features
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if contours:
        largest = max(contours, key=cv2.contourArea)
        M = cv2.moments(largest)
        area = M['m00'] / (edges.size + 1e-8)
        cx = (M['m10'] / (M['m00'] + 1e-8)) / (edges.shape[1] + 1e-8)
        cy = (M['m01'] / (M['m00'] + 1e-8)) / (edges.shape[0] + 1e-8)
        x, y, bw, bh = cv2.boundingRect(largest)
        aspect = bw / (bh + 1e-8)
    else:
        area, cx, cy, aspect = 0.0, 0.5, 0.5, 1.0
    
    # Feature vector (10-D)
    features = np.array([
        edges.mean() / 255.0,        # edge density
        edges.std() / 255.0,         # edge variance
        float(normalized.mean()),    # brightness
        float(normalized.std()),     # brightness variance
        edges.sum() / 255.0,         # total edge energy
        float(gray.mean()),          # gray intensity
        area,                        # blob area
        cx,                          # centroid x
        cy,                          # centroid y
        aspect,                      # aspect ratio
    ], dtype=np.float32)
    
    return dict(edges=edges, grayscale=gray, normalized=normalized, features=features)

print('✓ Image preprocessing function defined')

✓ Image preprocessing function defined


In [13]:
# Process all 11 frames
print('\n=== IMAGE PROCESSING ===\n')
processed_frames = []
img_features = []

for i, frame in enumerate(frames_raw):
    if TARGET_W and TARGET_H:
        frame = cv2.resize(frame, (TARGET_W, TARGET_H))
    
    result = preprocess_image(frame)
    processed_frames.append(result)
    img_features.append(result['features'])
    
    # Save edge image
    out_path = f"{OUTPUT_FOLDER}/frames_edges/edges_{i:02d}_t{i}s.png"
    cv2.imwrite(out_path, result['edges'])
    
    print(f'Frame {i:02d}  edges={result["edges"].mean():.4f}  '
          f'brightness={result["normalized"].mean():.4f}')

img_features = np.stack(img_features)  # (11, 10)
print(f'\n✓ Image features: {img_features.shape}')


=== IMAGE PROCESSING ===

Frame 00  edges=97.0677  brightness=0.5270
Frame 01  edges=96.7838  brightness=0.5194
Frame 02  edges=95.8342  brightness=0.5007
Frame 03  edges=96.1214  brightness=0.5753
Frame 04  edges=96.7273  brightness=0.5277
Frame 05  edges=96.6037  brightness=0.4733
Frame 06  edges=97.2827  brightness=0.4925
Frame 07  edges=96.7589  brightness=0.4998
Frame 08  edges=96.6925  brightness=0.4922
Frame 09  edges=96.6667  brightness=0.4907
Frame 10  edges=96.9390  brightness=0.4899

✓ Image features: (11, 10)


---
## 5. Audio Preprocessing Pipeline (OpenCV / NumPy DSP)

### Engraved Plan

| Step | Operation | Implementation | Purpose |
|------|-----------|----------------|---------|
| 1 | **Noise Reduction** | Spectral floor gate: zero DFT bins whose magnitude < `mean + 0.5 σ` | Remove broadband background hiss |
| 2 | **Silence Removal** | Short-time RMS per frame; mask frames below −40 dB threshold | Eliminate gaps between pledge lines |
| 3 | **Normalization** | Peak amplitude: `audio / max(|audio|)` | Bring amplitude to [−1, 1] regardless of mic gain |
| 4 | **Resampling** | Polyphase-equivalent via `numpy.interp` on fractional indices | Standardise to `TARGET_SR` (22 050 Hz) |
| 5 | **Segmentation** | Split into 11 fixed-length windows aligned to image timestamps | Lock audio and visual to the same 1-second epoch |
| 6 | **MFCC** | STFT → mel filterbank → log → DCT (all numpy.fft / numpy.linalg) | Compact spectral envelope descriptor for speech |
| 7 | **Spectrogram** | STFT magnitude converted to dB scale | Time-frequency map for visual inspection |
| 8 | **Chroma** | Map STFT bins to 12 pitch classes with equal-tempered filterbank | Tonal/melodic accent in the recitation |

In [14]:
def preprocess_audio(audio: np.ndarray, sr: int) -> dict:
    """
    Audio preprocessing using librosa:
    1. Trim silence
    2. Normalize
    3. Resample to target SR
    4. Extract MFCC
    5. Extract spectral features
    """
    # Step 1: Trim silence
    audio_trimmed, _ = librosa.effects.trim(audio, top_db=40)
    
    # Step 2: Normalize
    audio_norm = librosa.util.normalize(audio_trimmed)
    
    # Step 3: Resample
    if sr != TARGET_SR:
        audio_resampled = librosa.resample(audio_norm, orig_sr=sr, target_sr=TARGET_SR)
    else:
        audio_resampled = audio_norm
    
    sr_out = TARGET_SR
    
    # Step 4: MFCC features
    mfcc = librosa.feature.mfcc(y=audio_resampled, sr=sr_out, n_mfcc=N_MFCC)
    
    # Step 5: Spectral features
    spec = np.abs(librosa.stft(audio_resampled))
    spec_db = librosa.amplitude_to_db(spec, ref=np.max)
    
    # Step 6: Chroma features
    chroma = librosa.feature.chroma_cqt(y=audio_resampled, sr=sr_out)
    
    # Step 7: Zero-crossing rate
    zcr = librosa.feature.zero_crossing_rate(audio_resampled)
    
    # Feature vector (7-D)
    features = np.array([
        float(mfcc.mean()),           # mean MFCC
        float(mfcc.std()),            # MFCC variance
        float(spec_db.mean()),        # mean spectral power
        float(spec_db.std()),         # spectral variance
        float(chroma.mean()),         # mean chroma
        float(zcr.mean()),            # zero-crossing rate
        float(chroma.std()),          # chroma variance
    ], dtype=np.float32)
    
    return dict(
        audio=audio_resampled,
        sr=sr_out,
        mfcc=mfcc,
        spectrogram=spec_db,
        chroma=chroma,
        zcr=zcr,
        features=features
    )

print('✓ Audio preprocessing function defined')

✓ Audio preprocessing function defined


In [15]:
def segment_audio(audio: np.ndarray, n_segments: int = NUM_FRAMES):
    """Divide audio into equal-length segments"""
    seg_len = len(audio) // n_segments
    segments = []
    for i in range(n_segments):
        start = i * seg_len
        end = (i + 1) * seg_len if i < n_segments - 1 else len(audio)
        segments.append(audio[start:end])
    return segments

# Full-clip preprocessing
print('Processing full audio clip...')
audio_full = preprocess_audio(audio_raw, sr_orig)
audio_clean = audio_full['audio']

# Segment into 11 windows
segments = segment_audio(audio_clean, NUM_FRAMES)
print(f'✓ Segmented into {len(segments)} windows')

Processing full audio clip...
✓ Segmented into 11 windows


---
## 6. Segmentation — 11 Audio Windows Aligned to Image Timestamps

In [16]:
def segment_audio(audio: np.ndarray, sr: int,
                  n_segments: int = NUM_FRAMES,
                  total_sec: int  = VIDEO_SECONDS) -> list[dict]:
    """
    Divide audio into `n_segments` equal-length windows covering `total_sec`.
    Returns a list of dicts, each with keys: audio, t_start, t_end.
    """
    seg_len_samples = len(audio) / n_segments
    sec_per_seg     = total_sec / n_segments
    segments = []
    for i in range(n_segments):
        start = int(i * seg_len_samples)
        end   = int((i + 1) * seg_len_samples)
        chunk = audio[start:end]
        segments.append(dict(
            audio   = chunk,
            t_start = round(i * sec_per_seg, 3),
            t_end   = round((i + 1) * sec_per_seg, 3),
        ))
    return segments


# ── Step 1-3: full-clip preprocessing ────────────────────────────────────────
print('Running full-clip audio preprocessing …')
audio_full_result = preprocess_audio(audio_raw, sr_orig)
audio_clean       = audio_full_result['resampled']   # normalised, resampled

# ── Step 4: segmentation ─────────────────────────────────────────────────────
segments_raw = segment_audio(audio_clean, TARGET_SR)
print(f'\n✓ Segmented into {len(segments_raw)} windows:')
for seg in segments_raw:
    print(f'  {seg["t_start"]:.2f}s – {seg["t_end"]:.2f}s  '
          f'({len(seg["audio"])} samples)')

Running full-clip audio preprocessing …


KeyError: 'resampled'

In [ ]:
print('\n=== AUDIO PROCESSING ===\n')
processed_audio = []
aud_features = []

for i, segment in enumerate(segments):
    result = preprocess_audio(segment, TARGET_SR)
    processed_audio.append(result)
    aud_features.append(result['features'])
    
    # Save audio segment as WAV
    seg_path = f"{OUTPUT_FOLDER}/audio_segments/segment_{i:02d}_t{i}s.wav"
    librosa.output.write_wav(seg_path, result['audio'], sr=result['sr'])
    
    print(f'Segment {i:02d}  MFCC_μ={result["features"][0]:.4f}  '
          f'spec_μ={result["features"][2]:.4f}')

aud_features = np.stack(aud_features)  # (11, 7)
print(f'\n✓ Audio features: {aud_features.shape}')

---
## 7. Feature Fusion — Tandem Pairing (Image ↔ Audio)

Each of the 11 time-steps fuses its image feature vector (10-D) with the
audio feature vector (7-D) into a combined 17-D representation.

In [ ]:
print('\n=== FEATURE FUSION ===\n')

# Fuse image + audio features (10-D + 7-D = 17-D)
fused_features = np.hstack([img_features, aud_features])
timeline_rows = []

print(f'{"Frame":>5}  {"EdgeDens":>8}  {"Bright":>8}  {"MFCC":>8}  {"Spec":>8}')
print('-' * 50)

for i in range(NUM_FRAMES):
    row = {
        'frame': i,
        'timestamp': i,
        'edge_density': float(img_features[i, 0]),
        'brightness': float(img_features[i, 2]),
        'mfcc_mean': float(aud_features[i, 0]),
        'spec_mean': float(aud_features[i, 2]),
        'zcr': float(aud_features[i, 5]),
    }
    timeline_rows.append(row)
    
    print(f'{i:5d}  {row["edge_density"]:8.4f}  {row["brightness"]:8.4f}  '
          f'{row["mfcc_mean"]:8.4f}  {row["spec_mean"]:8.4f}')

print(f'\n✓ Fused features: {fused_features.shape}')

---
## 8. Save Outputs

In [ ]:
# Save feature arrays
np.save(f'{OUTPUT_FOLDER}/features/img_features.npy', img_features)
np.save(f'{OUTPUT_FOLDER}/features/aud_features.npy', aud_features)
np.save(f'{OUTPUT_FOLDER}/features/fused_features.npy', fused_features)

# Save MFCC per segment
mfcc_all = np.stack([r['mfcc'].mean(axis=1) for r in processed_audio])
np.save(f'{OUTPUT_FOLDER}/features/mfcc_per_segment.npy', mfcc_all)

# Save timeline CSV
csv_path = f'{OUTPUT_FOLDER}/features/confidence_timeline.csv'
fieldnames = list(timeline_rows[0].keys())
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(timeline_rows)

print('\n✓ Saved outputs:')
print(f'  {OUTPUT_FOLDER}/features/img_features.npy')
print(f'  {OUTPUT_FOLDER}/features/aud_features.npy')
print(f'  {OUTPUT_FOLDER}/features/fused_features.npy')
print(f'  {OUTPUT_FOLDER}/features/mfcc_per_segment.npy')
print(f'  {csv_path}')

---
## 9. Inline Visualisation (OpenCV-rendered, no matplotlib)

Writes diagnostic grid images to disk and displays them via `cv2.imshow`
when running locally, or skips display in headless environments.

In [ ]:
import matplotlib.pyplot as plt

# Visualize spectrogram
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

# Spectrogram
spec_db = audio_full['spectrogram']
img = librosa.display.specshow(spec_db, sr=TARGET_SR, x_axis='time', y_axis='log', ax=axes[0])
axes[0].set_title('Spectrogram (dB)')
fig.colorbar(img, ax=axes[0])

# MFCC
mfcc = audio_full['mfcc']
img = librosa.display.specshow(mfcc, sr=TARGET_SR, x_axis='time', ax=axes[1])
axes[1].set_title('MFCC Features')
fig.colorbar(img, ax=axes[1])

plt.tight_layout()
plt.savefig(f'{OUTPUT_FOLDER}/features/audio_features.png', dpi=100, bbox_inches='tight')
print(f'✓ Saved audio visualization → {OUTPUT_FOLDER}/features/audio_features.png')
plt.show()

# Create edge grid
fig, axes = plt.subplots(3, 4, figsize=(14, 8))
axes = axes.flatten()
for i, result in enumerate(processed_frames[:NUM_FRAMES]):
    axes[i].imshow(result['edges'], cmap='gray')
    axes[i].set_title(f'Frame {i}')
    axes[i].axis('off')
if NUM_FRAMES < len(axes):
    for i in range(NUM_FRAMES, len(axes)):
        axes[i].axis('off')

plt.tight_layout()
plt.savefig(f'{OUTPUT_FOLDER}/features/edge_grid.png', dpi=100, bbox_inches='tight')
print(f'✓ Saved edge grid → {OUTPUT_FOLDER}/features/edge_grid.png')
plt.show()

print('\n✓ All visualizations complete!')

---
## Summary

**Pipeline Overview:**

| Stage | Function | Output |
|-------|----------|--------|
| Image Blur | `cv2.GaussianBlur` | noise-reduced BGR |
| Grayscale | `cv2.cvtColor` | single-channel uint8 |
| Normalize | Min-Max scaling | float32 [0,1] |
| Edge Detection | `cv2.Canny` | binary edge map |
| Shape Features | `cv2.findContours` + moments | area, centroid, aspect |
| Audio Trim | `librosa.effects.trim` | silence removed |
| Audio Normalize | `librosa.util.normalize` | amplitude normalized |
| Resample | `librosa.resample` | 22050 Hz |
| MFCC | `librosa.feature.mfcc` | (13, T) features |
| Spectrogram | `librosa.stft` → dB | (F, T) magnitude |
| Chroma | `librosa.feature.chroma_cqt` | (12, T) pitch features |
| Fusion | `np.hstack` | 17-D vector per frame |

**Output Files:**
- `img_features.npy` - (11, 10) image features
- `aud_features.npy` - (11, 7) audio features
- `fused_features.npy` - (11, 17) combined features
- `confidence_timeline.csv` - frame-by-frame timeline
- `frames_edges/` - 11 Canny edge PNGs
- `audio_segments/` - 11 WAV files (one per second)
- `audio_features.png` - Spectrogram + MFCC visualization
- `edge_grid.png` - Grid of all edge frames